# Strict Bottom-Code CAMPA v2 for DoAO

This notebook extends the **successful strict bottom-codebook experiment** while preserving the exact no-SAM Christian evaluation protocol that previously produced a mean IoU of approximately **0.337824**.

## What is unchanged

- Training uses only consecutive **after/legal** frames.
- Christian's VQ-VAE-2 checkpoint initializes every run.
- Matching uses **bottom-level quantized codebook vectors only**.
- Christian's reconstruction, crop, consistency, and VQ losses remain present.
- Final anomaly detection uses **reconstruction error only**. Memory maps remain diagnostics.
- Threshold calibration, mask loading, morphology, and score precision are copied from the successful strict experiment.

## What is changed in v2

1. **Two-frame memory (`T=2`)**: each current patch searches the two immediately preceding legal frames.
2. **Two matches total (`K=2`)**: the two best vectors are selected across both frames and all local candidates—not two per frame.
3. **Temporal decay**: recent frames receive greater affinity, following CAMPA's memory principle.
4. **Better confidence**: confidence uses absolute similarity plus the top-1/top-2 margin; equal matches with the same code ID are not treated as ambiguous.
5. **Sparse code-identity consistency**: the propagated previous code IDs form a soft categorical target for the current patch's differentiable code assignment.
6. **Clean-after tail regularization**: a small penalty discourages unusually high reconstruction errors on legal after pixels.

## Relationship to MemCAM

This is a faithful adaptation of **CAMPA's propagation/self-supervision principle**, not an exact reproduction of the full MemCAM network.

Directly retained from CAMPA:

- a memory bank of previous frames;
- cosine similarity between current and past patches;
- temporal decay;
- top-K selection;
- temperature-scaled softmax;
- similarity-weighted propagation;
- a propagated synthetic target used through consistency losses.

Adapted for DoAO:

- VQ-VAE bottom code vectors replace transformer patch embeddings;
- propagated code vectors/code identities replace propagated CAM values;
- cosine and sparse categorical consistency replace CAM-map KL divergence;
- local search replaces global matching for computational and conveyor-motion reasons;
- there is no DeiT/MCTFormer, class CAM, video label, or original trainable MAM block.

The accurate name is **MemCAM/CAMPA-inspired bottom-code propagation**, not “the original MemCAM model.”


In [1]:
# ============================================================
# 0) Imports and configuration
# ============================================================

import os
import sys
import re
import gc
import json
import time
import math
import random
import shutil
import zipfile
import hashlib
import subprocess
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from scipy.ndimage import (
    gaussian_filter,
    binary_closing,
    binary_opening,
    binary_fill_holes,
    generate_binary_structure,
    label as cc_label,
)
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
plt.switch_backend("Agg")


# Kaggle paths
GITHUB_REPO = "https://github.com/Shuvro-Ahmed/Anamoly-waste-extension-polimi.git"
CHRISTIAN_CODE_INPUT = "/kaggle/input/models/shuvroahmedimaging/christian/other/default/1"
DATASET_ROOT = Path("/kaggle/input/datasets/shuvroahmedimaging/ws2-eca")

OUT_DIR = Path("/kaggle/working/strict_bottom_code_campa_v2")
CKPT_DIR = OUT_DIR / "checkpoints"
VIS_DIR = OUT_DIR / "visuals"
for directory in [OUT_DIR, CKPT_DIR, VIS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FINAL_ZIP = Path("/kaggle/working/strict_bottom_code_campa_v2_outputs.zip")


# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    if torch.cuda.get_device_capability(0)[0] < 7:
        raise RuntimeError(
            "Use Kaggle GPU T4 x2. The current PyTorch image may not support P100/sm_60."
        )
print("DEVICE:", DEVICE)


# Image protocol
IMG_SIZE_TRAIN = 512
MODEL_SIZE = 256
IMG_SIZE_INFER = 256
CROP_SIZE = 256


# Training
EPOCHS = 10
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = False
PERSISTENT_WORKERS = False
BASE_SEED = 1337
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-6
LR_VQVAE = 3e-5
REUSE_EXISTING_CHECKPOINTS = False


# Strict code-memory settings
SEARCH_RADIUS = 4
TOP_K = 2
ATTN_TEMPERATURE = 0.08
SPATIAL_BIAS = 0.04
TEMPORAL_DECAY = 0.90

# New confidence calibration
CONF_ABS_CENTER = 0.35
CONF_ABS_SCALE = 0.10
CONF_MARGIN_CENTER = 0.015
CONF_MARGIN_SCALE = 0.015

# Differentiable current code assignment for L_code
CODE_ASSIGN_TEMPERATURE = 0.20
CODE_LOGIT_CHUNK = 4096

# Legal reconstruction-tail preservation
CLEAN_TAIL_QUANTILE = 0.95
MAX_CLEAN_CALIBRATION_FRAMES = 600

# Christian loss coefficients from the released direct-sum implementation
LAMBDA_REC = 1.0
LAMBDA_CROPS = 1.0
LAMBDA_CONS = 1.0
LAMBDA_VQ = 1.0


# Independent runs. All restart from Christian's checkpoint.
RUN_CONFIGS = [
    {
        "name": "continued_baseline_corrected_crop",
        "kind": "baseline",
        "memory_frames": 0,
        "confidence_mode": "none",
        "lambda_qmem": 0.0,
        "lambda_code": 0.0,
        "lambda_clean": 0.0,
    },
    {
        "name": "strict_qmem_v1_reference_T1K2_lambda0p05",
        "kind": "qmem",
        "memory_frames": 1,
        "confidence_mode": "legacy",
        "lambda_qmem": 0.05,
        "lambda_code": 0.0,
        "lambda_clean": 0.0,
    },
    {
        "name": "strict_qmem_v2_T2K2_qmem_only_lambda0p02",
        "kind": "qmem",
        "memory_frames": 2,
        "confidence_mode": "margin",
        "lambda_qmem": 0.02,
        "lambda_code": 0.0,
        "lambda_clean": 0.0,
    },
    {
        "name": "strict_qmem_v2_T2K2_qmem_only_lambda0p05",
        "kind": "qmem",
        "memory_frames": 2,
        "confidence_mode": "margin",
        "lambda_qmem": 0.05,
        "lambda_code": 0.0,
        "lambda_clean": 0.0,
    },
    {
        "name": "strict_qmem_v2_T2K2_qmem_code",
        "kind": "qmem",
        "memory_frames": 2,
        "confidence_mode": "margin",
        "lambda_qmem": 0.05,
        "lambda_code": 1e-4,
        "lambda_clean": 0.0,
    },
    {
        "name": "strict_qmem_v2_T2K2_full",
        "kind": "qmem",
        "memory_frames": 2,
        "confidence_mode": "margin",
        "lambda_qmem": 0.05,
        "lambda_code": 1e-4,
        "lambda_clean": 0.10,
    },
]


# Evaluation locked to the successful strict notebook
K_TUKEY = 4.0
MORPH_MIN_COMPONENT = 50
GAUSSIAN_SIGMA = 1.0

EXPECTED_CHRISTIAN_IOU = 0.337824432061603
BASELINE_IOU_TOLERANCE = 0.002
EXPECTED_CHRISTIAN_THRESHOLD = 0.1355597823858261
BASELINE_THRESHOLD_TOLERANCE = 0.003

MAX_TRAIN_SEQUENCES = None
MAX_VAL_SEQUENCES = None
MAX_THRESHOLD_FRAMES = None
MAX_EVAL_FRAMES = None

RUN_TRAINING = True
RUN_EVALUATION = True
RUN_VISUALS = True

print("RUN_CONFIGS:")
for config in RUN_CONFIGS:
    print(config)


GPU: Tesla T4
CUDA capability: (7, 5)
DEVICE: cuda
RUN_CONFIGS:
{'name': 'continued_baseline_corrected_crop', 'kind': 'baseline', 'memory_frames': 0, 'confidence_mode': 'none', 'lambda_qmem': 0.0, 'lambda_code': 0.0, 'lambda_clean': 0.0}
{'name': 'strict_qmem_v1_reference_T1K2_lambda0p05', 'kind': 'qmem', 'memory_frames': 1, 'confidence_mode': 'legacy', 'lambda_qmem': 0.05, 'lambda_code': 0.0, 'lambda_clean': 0.0}
{'name': 'strict_qmem_v2_T2K2_qmem_only_lambda0p02', 'kind': 'qmem', 'memory_frames': 2, 'confidence_mode': 'margin', 'lambda_qmem': 0.02, 'lambda_code': 0.0, 'lambda_clean': 0.0}
{'name': 'strict_qmem_v2_T2K2_qmem_only_lambda0p05', 'kind': 'qmem', 'memory_frames': 2, 'confidence_mode': 'margin', 'lambda_qmem': 0.05, 'lambda_code': 0.0, 'lambda_clean': 0.0}
{'name': 'strict_qmem_v2_T2K2_qmem_code', 'kind': 'qmem', 'memory_frames': 2, 'confidence_mode': 'margin', 'lambda_qmem': 0.05, 'lambda_code': 0.0001, 'lambda_clean': 0.0}
{'name': 'strict_qmem_v2_T2K2_full', 'kind': 'qmem

In [2]:
# ============================================================
# 1) Load Christian's exact architecture and checkpoint
# ============================================================

WORK_ROOT = Path("/kaggle/working/christian_code_strict_campa_v2")
shutil.rmtree(WORK_ROOT, ignore_errors=True)


def find_code_dir(root):
    matches = list(Path(root).rglob("proposed_solution"))
    if not matches:
        raise FileNotFoundError(f"Could not find proposed_solution under {root}")
    return matches[0].parent


if Path(CHRISTIAN_CODE_INPUT).exists():
    shutil.copytree(CHRISTIAN_CODE_INPUT, WORK_ROOT, dirs_exist_ok=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO, str(WORK_ROOT)], check=True)

CODE_DIR = find_code_dir(WORK_ROOT)
PROPOSED_DIR = CODE_DIR / "proposed_solution"
sys.path.insert(0, str(CODE_DIR))

from proposed_solution.architecture.model import VQVAE2
from proposed_solution.utils import get_corner_crops

CHRISTIAN_CKPT = PROPOSED_DIR / "checkpoint" / "model.pth"
if not CHRISTIAN_CKPT.exists():
    raise FileNotFoundError(f"Christian checkpoint not found: {CHRISTIAN_CKPT}")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


probe = VQVAE2()
required = [
    "encoder_bottom",
    "encoder_top",
    "quantizer_top",
    "quantizer_bottom",
    "pre_quant_bottom",
    "decoder_top",
    "decoder_final",
]
missing = [name for name in required if not hasattr(probe, name)]
del probe
if missing:
    raise RuntimeError(f"Christian VQVAE2 is missing required attributes: {missing}")

print("CODE_DIR:", CODE_DIR)
print("CHRISTIAN_CKPT:", CHRISTIAN_CKPT)
print("CHECKPOINT_SHA256:", sha256_file(CHRISTIAN_CKPT))


CODE_DIR: /kaggle/working/christian_code_strict_campa_v2/Code
CHRISTIAN_CKPT: /kaggle/working/christian_code_strict_campa_v2/Code/proposed_solution/checkpoint/model.pth
CHECKPOINT_SHA256: 18d0628f3dacc0739d604465aa64961d6a889c8f05f6a863048efffddc8520a0


In [3]:
# ============================================================
# 2) Official WS2 splits and exact crop reassembly
# ============================================================


def natural_key(path):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(path))]


def list_images(folder, recursive=False):
    folder = Path(folder)
    files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
        files.extend(folder.rglob(ext) if recursive else folder.glob(ext))
    return sorted(files, key=natural_key)


def count_images_recursive(folder):
    return len(list_images(folder, recursive=True))


def find_ws2_splits(dataset_root):
    dataset_root = Path(dataset_root)

    train_val_candidates = []
    for after_dir in dataset_root.rglob("training/after"):
        root = after_dir.parents[1]
        if (root / "validation" / "after").exists():
            train_val_candidates.append(root)
    if not train_val_candidates:
        raise FileNotFoundError("Could not find training/after and validation/after.")

    train_val_root = sorted(
        train_val_candidates,
        key=lambda p: (0 if "with_background" in str(p).lower() else 1, str(p)),
    )[0]

    test_candidates = []
    for mask_before in dataset_root.rglob("masks/before"):
        root = mask_before.parents[1]
        if (root / "images" / "before").exists():
            test_candidates.append(root)
    if not test_candidates:
        raise FileNotFoundError("Could not find test_set/images/before and masks/before.")

    test_root = sorted(
        test_candidates,
        key=lambda p: (0 if "test_set" in str(p).lower() else 1, str(p)),
    )[0]

    return {
        "train_after": train_val_root / "training" / "after",
        "val_after": train_val_root / "validation" / "after",
        "test_before": test_root / "images" / "before",
        "test_masks": test_root / "masks" / "before",
    }


SPLITS = find_ws2_splits(DATASET_ROOT)
for key, value in SPLITS.items():
    print(key, value, "images:", count_images_recursive(value))

TRAIN_AFTER_ROOT = SPLITS["train_after"]
VAL_AFTER_ROOT = SPLITS["val_after"]
BEFORE_ROOT = SPLITS["test_before"]
BEFORE_MASK_ROOT = SPLITS["test_masks"]


def reassemble_crops_to_full_fixed(crops, full_size):
    """Place four reconstructed crops into the correct 512x512 quadrants."""
    b4, channels, _, _ = crops.shape
    if b4 % 4 != 0:
        raise ValueError("Expected four crops per image.")

    batch = b4 // 4
    half = full_size // 2
    crops = F.interpolate(crops, size=(half, half), mode="bilinear", align_corners=False)

    result = torch.zeros(
        batch,
        channels,
        full_size,
        full_size,
        device=crops.device,
        dtype=crops.dtype,
    )
    result[:, :, :half, :half] = crops[0::4]
    result[:, :, :half, half:] = crops[1::4]
    result[:, :, half:, :half] = crops[2::4]
    result[:, :, half:, half:] = crops[3::4]
    return result


train_after /kaggle/input/datasets/shuvroahmedimaging/ws2-eca/train_val_with_backgrounds/train_val_with_backgrounds/training/after images: 3829
val_after /kaggle/input/datasets/shuvroahmedimaging/ws2-eca/train_val_with_backgrounds/train_val_with_backgrounds/validation/after images: 883
test_before /kaggle/input/datasets/shuvroahmedimaging/ws2-eca/test_set/test_set/images/before images: 496
test_masks /kaggle/input/datasets/shuvroahmedimaging/ws2-eca/test_set/test_set/masks/before images: 496


In [4]:
# ============================================================
# 3) Consecutive after-frame sequence datasets
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_TRAIN, IMG_SIZE_TRAIN)),
    transforms.ToTensor(),
])


class ConsecutiveAfterSequences(Dataset):
    """Return T previous after frames and one current after frame from one video."""

    def __init__(self, root, memory_frames, max_sequences=None):
        self.root = Path(root)
        self.memory_frames = int(memory_frames)
        self.samples = []

        for video_dir in sorted([p for p in self.root.iterdir() if p.is_dir()], key=natural_key):
            frames = list_images(video_dir)
            for current_index in range(self.memory_frames, len(frames)):
                previous_paths = frames[current_index - self.memory_frames:current_index]
                current_path = frames[current_index]
                self.samples.append((previous_paths, current_path, video_dir.name))

        if max_sequences is not None:
            self.samples = self.samples[:int(max_sequences)]

        if not self.samples:
            raise RuntimeError(f"No sequences with T={self.memory_frames} found under {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        previous_paths, current_path, video_name = self.samples[index]
        previous = torch.stack([
            train_transform(Image.open(path).convert("RGB"))
            for path in previous_paths
        ])
        current = train_transform(Image.open(current_path).convert("RGB"))
        return previous, current, [str(path) for path in previous_paths], str(current_path), video_name


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def collate_sequences(batch):
    previous = torch.stack([item[0] for item in batch])
    current = torch.stack([item[1] for item in batch])
    previous_paths = [item[2] for item in batch]
    current_paths = [item[3] for item in batch]
    video_names = [item[4] for item in batch]
    return previous, current, previous_paths, current_paths, video_names


_DATASET_CACHE = {}


def dataset_for_t(root, memory_frames, max_sequences):
    key = (str(root), int(memory_frames), max_sequences)
    if key not in _DATASET_CACHE:
        _DATASET_CACHE[key] = ConsecutiveAfterSequences(
            root,
            memory_frames=int(memory_frames),
            max_sequences=max_sequences,
        )
    return _DATASET_CACHE[key]


def make_train_loader(seed, memory_frames):
    dataset = dataset_for_t(TRAIN_AFTER_ROOT, memory_frames, MAX_TRAIN_SEQUENCES)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
        worker_init_fn=seed_worker,
        generator=generator,
        collate_fn=collate_sequences,
    )


def make_val_loader(memory_frames):
    dataset = dataset_for_t(VAL_AFTER_ROOT, memory_frames, MAX_VAL_SEQUENCES)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
        worker_init_fn=seed_worker,
        collate_fn=collate_sequences,
    )


for t in sorted({max(1, int(config["memory_frames"])) for config in RUN_CONFIGS}):
    print(
        f"T={t} training sequences:",
        len(dataset_for_t(TRAIN_AFTER_ROOT, t, MAX_TRAIN_SEQUENCES)),
        "validation sequences:",
        len(dataset_for_t(VAL_AFTER_ROOT, t, MAX_VAL_SEQUENCES)),
    )


T=1 training sequences: 3589 validation sequences: 851
T=2 training sequences: 3349 validation sequences: 819


In [5]:
# ============================================================
# 4) Model loading, quantization helpers, clean-tail calibration
# ============================================================


def load_state_dict_safely(path):
    try:
        state = torch.load(str(path), map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(str(path), map_location=DEVICE)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    return {
        key[len("module."):] if key.startswith("module.") else key: value
        for key, value in state.items()
    }


def make_base_model(checkpoint):
    model = VQVAE2().to(DEVICE)
    model.load_state_dict(load_state_dict_safely(checkpoint))
    return model


def nearest_codebook_no_update(z, quantizer):
    """Nearest-codebook quantization without EMA-buffer updates."""
    z_perm = z.permute(0, 2, 3, 1).contiguous()
    flat = z_perm.view(-1, quantizer.embedding_dim)
    embedding = quantizer.embedding

    distances = (
        flat.square().sum(dim=1, keepdim=True)
        - 2.0 * (flat @ embedding.t())
        + embedding.square().sum(dim=1).unsqueeze(0)
    )
    indices = distances.argmin(dim=1)

    quantized = F.embedding(indices, embedding)
    quantized = quantized.view(z_perm.shape).permute(0, 3, 1, 2).contiguous()
    indices = indices.view(z.shape[0], z.shape[2], z.shape[3])
    return quantized, indices


def shift_candidate(x, dy, dx):
    """Return x[y+dy,x+dx] aligned to the current-grid coordinates."""
    batch, channels, height, width = x.shape
    shifted = torch.zeros_like(x)
    valid = torch.zeros(batch, 1, height, width, device=x.device, dtype=x.dtype)

    out_y0, out_y1 = max(0, -dy), min(height, height - dy)
    out_x0, out_x1 = max(0, -dx), min(width, width - dx)
    if out_y1 <= out_y0 or out_x1 <= out_x0:
        return shifted, valid

    src_y0, src_y1 = out_y0 + dy, out_y1 + dy
    src_x0, src_x1 = out_x0 + dx, out_x1 + dx
    shifted[:, :, out_y0:out_y1, out_x0:out_x1] = x[
        :, :, src_y0:src_y1, src_x0:src_x1
    ]
    valid[:, :, out_y0:out_y1, out_x0:out_x1] = 1.0
    return shifted, valid


@torch.no_grad()
def calibrate_clean_tail_threshold():
    """High legal-error quantile from untouched-Christian training-after images."""
    output_file = OUT_DIR / "clean_tail_calibration.json"
    if output_file.exists():
        return json.loads(output_file.read_text())

    paths = list_images(TRAIN_AFTER_ROOT, recursive=True)
    rng = np.random.default_rng(BASE_SEED + 71)
    if len(paths) > MAX_CLEAN_CALIBRATION_FRAMES:
        indices = rng.choice(len(paths), size=MAX_CLEAN_CALIBRATION_FRAMES, replace=False)
        paths = [paths[int(i)] for i in sorted(indices)]

    model = make_base_model(CHRISTIAN_CKPT).eval()
    transform = transforms.Compose([
        transforms.Resize((MODEL_SIZE, MODEL_SIZE)),
        transforms.ToTensor(),
    ])

    samples = []
    for path in tqdm(paths, desc="calibrate clean-tail threshold"):
        x = transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
        recon, _ = model(x)
        error = torch.mean(torch.abs(x - recon), dim=1)
        samples.append(error[0].cpu().numpy().reshape(-1)[::4])

    values = np.concatenate(samples)
    threshold = float(np.quantile(values, CLEAN_TAIL_QUANTILE))
    result = {
        "quantile": float(CLEAN_TAIL_QUANTILE),
        "threshold": threshold,
        "n_frames": int(len(paths)),
        "n_sampled_pixels": int(len(values)),
    }
    output_file.write_text(json.dumps(result, indent=2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result


CLEAN_TAIL_STATS = calibrate_clean_tail_threshold()
CLEAN_TAIL_THRESHOLD = float(CLEAN_TAIL_STATS["threshold"])
print("CLEAN_TAIL_STATS:", CLEAN_TAIL_STATS)


calibrate clean-tail threshold:   0%|          | 0/600 [00:00<?, ?it/s]

CLEAN_TAIL_STATS: {'quantile': 0.95, 'threshold': 0.1425803005695343, 'n_frames': 600, 'n_sampled_pixels': 9830400}


In [6]:
# ============================================================
# 5) Strict bottom-code CAMPA module
# ============================================================


class StrictBottomCodeCAMPA(nn.Module):
    """CAMPA-style propagation over bottom quantized VQ-VAE code vectors."""

    def __init__(self, base_model, memory_frames, confidence_mode="margin"):
        super().__init__()
        self.base = base_model
        self.memory_frames = int(memory_frames)
        self.confidence_mode = str(confidence_mode)
        self.offsets = [
            (dy, dx)
            for dy in range(-SEARCH_RADIUS, SEARCH_RADIUS + 1)
            for dx in range(-SEARCH_RADIUS, SEARCH_RADIUS + 1)
        ]
        candidate_count = self.memory_frames * len(self.offsets)
        if TOP_K > candidate_count:
            raise ValueError(f"TOP_K={TOP_K} exceeds {candidate_count} candidates")

    def encode_current(self, image):
        base = self.base
        bottom = base.encoder_bottom(image)
        top = base.encoder_top(bottom)

        q_top_st, q_loss_top, perplexity_top = base.quantizer_top(top)
        top_context = base.decoder_top(q_top_st)
        if top_context.shape[2:] != bottom.shape[2:]:
            top_context = F.interpolate(top_context, size=bottom.shape[2:], mode="nearest")

        prequant_bottom = base.pre_quant_bottom(torch.cat([bottom, top_context], dim=1))
        q_bottom_st, q_loss_bottom, perplexity_bottom = base.quantizer_bottom(prequant_bottom)
        reconstruction = torch.sigmoid(
            base.decoder_final(torch.cat([q_bottom_st, top_context], dim=1))
        )

        return {
            "reconstruction": reconstruction,
            "prequant_bottom": prequant_bottom,
            "q_bottom_st": q_bottom_st,
            "vq_loss": q_loss_top + q_loss_bottom,
            "perplexity_top": perplexity_top,
            "perplexity_bottom": perplexity_bottom,
        }

    @torch.no_grad()
    def encode_previous_codes(self, previous_images):
        if previous_images.ndim != 5:
            raise ValueError(f"Expected B,T,C,H,W, got {previous_images.shape}")

        batch, frames, channels, height, width = previous_images.shape
        flat = previous_images.view(batch * frames, channels, height, width)

        base = self.base
        bottom = base.encoder_bottom(flat)
        top = base.encoder_top(bottom)
        q_top, _ = nearest_codebook_no_update(top, base.quantizer_top)
        top_context = base.decoder_top(q_top)
        if top_context.shape[2:] != bottom.shape[2:]:
            top_context = F.interpolate(top_context, size=bottom.shape[2:], mode="nearest")

        prequant_bottom = base.pre_quant_bottom(torch.cat([bottom, top_context], dim=1))
        q_bottom, indices = nearest_codebook_no_update(prequant_bottom, base.quantizer_bottom)

        latent_channels = q_bottom.shape[1]
        latent_h, latent_w = q_bottom.shape[2:]
        q_bottom = q_bottom.view(batch, frames, latent_channels, latent_h, latent_w)
        indices = indices.view(batch, frames, latent_h, latent_w)
        return q_bottom.detach(), indices.detach()

    def _legacy_confidence(self, top_scores, selected_code_ids):
        attention = F.softmax(top_scores / ATTN_TEMPERATURE, dim=1)
        entropy = -(
            attention * torch.log(attention + 1e-8)
        ).sum(dim=1) / math.log(float(TOP_K))
        entropy_confidence = (1.0 - entropy).clamp(0.0, 1.0)
        similarity_confidence = torch.sigmoid((top_scores[:, 0] - 0.35) / 0.10)
        return (entropy_confidence * similarity_confidence).clamp(0.0, 1.0)

    def _margin_confidence(self, top_scores, selected_code_ids):
        best = top_scores[:, 0]
        second = top_scores[:, 1]
        margin = best - second

        absolute_confidence = torch.sigmoid((best - CONF_ABS_CENTER) / CONF_ABS_SCALE)
        margin_confidence = torch.sigmoid(
            (margin - CONF_MARGIN_CENTER) / CONF_MARGIN_SCALE
        )

        # Equal candidates are not ambiguous if they imply the same code target.
        same_code = selected_code_ids[:, 0] == selected_code_ids[:, 1]
        margin_confidence = torch.where(
            same_code,
            torch.ones_like(margin_confidence),
            margin_confidence,
        )
        return (absolute_confidence * margin_confidence).clamp(0.0, 1.0)

    def propagate_previous_codes(self, current_q_st, previous_q, previous_indices):
        current_for_match = F.normalize(current_q_st.detach(), dim=1)

        candidate_scores = []
        candidate_index_maps = []
        candidate_metadata = []
        radius_sq = max(float(SEARCH_RADIUS ** 2), 1.0)

        # previous_q ordering is oldest -> newest.
        for frame_position in range(previous_q.shape[1]):
            age = previous_q.shape[1] - frame_position
            temporal_factor = TEMPORAL_DECAY ** (age - 1)
            frame_q = previous_q[:, frame_position]
            frame_indices = previous_indices[:, frame_position]
            frame_match = F.normalize(frame_q.detach(), dim=1)

            for dy, dx in self.offsets:
                shifted_feature, valid = shift_candidate(frame_match, dy, dx)
                similarity = (current_for_match * shifted_feature).sum(dim=1)
                similarity = temporal_factor * similarity
                normalized_distance = float(dy * dy + dx * dx) / radius_sq
                similarity = similarity - SPATIAL_BIAS * normalized_distance
                similarity = similarity.masked_fill(valid[:, 0] < 0.5, -1e4)

                shifted_indices, _ = shift_candidate(
                    frame_indices.unsqueeze(1).float(), dy, dx
                )
                candidate_scores.append(similarity)
                candidate_index_maps.append(shifted_indices[:, 0].long())
                candidate_metadata.append({
                    "frame_position": frame_position,
                    "age": age,
                    "dy": dy,
                    "dx": dx,
                })

        scores = torch.stack(candidate_scores, dim=1)                 # B,P,H,W
        candidate_indices = torch.stack(candidate_index_maps, dim=1) # B,P,H,W
        top_scores, top_candidate_ids = torch.topk(scores, k=TOP_K, dim=1)
        attention = F.softmax(top_scores / ATTN_TEMPERATURE, dim=1)
        selected_code_ids = torch.gather(candidate_indices, 1, top_candidate_ids)

        propagated = torch.zeros_like(previous_q[:, -1])
        for candidate_id, metadata in enumerate(candidate_metadata):
            selected_weight = (
                attention * (top_candidate_ids == candidate_id).to(attention.dtype)
            ).sum(dim=1)
            frame_q = previous_q[:, metadata["frame_position"]]
            shifted_value, _ = shift_candidate(frame_q.detach(), metadata["dy"], metadata["dx"])
            propagated = propagated + selected_weight.unsqueeze(1) * shifted_value

        if self.confidence_mode == "legacy":
            confidence = self._legacy_confidence(top_scores, selected_code_ids)
        elif self.confidence_mode == "margin":
            confidence = self._margin_confidence(top_scores, selected_code_ids)
        else:
            raise ValueError(f"Unknown confidence mode: {self.confidence_mode}")

        mismatch = 1.0 - F.cosine_similarity(
            current_q_st, propagated.detach(), dim=1, eps=1e-8
        )
        confidence_detached = confidence.detach()
        qmem_loss = (
            confidence_detached * mismatch
        ).sum() / (confidence_detached.sum() + 1e-6)

        candidate_age_lookup = torch.tensor(
            [item["age"] for item in candidate_metadata],
            device=top_candidate_ids.device,
            dtype=torch.float32,
        )
        selected_ages = candidate_age_lookup[top_candidate_ids]
        mean_selected_age = (attention * selected_ages).sum(dim=1)
        same_code = (selected_code_ids[:, 0] == selected_code_ids[:, 1]).float()
        score_margin = top_scores[:, 0] - top_scores[:, 1]

        diagnostics = {
            "memory": propagated,
            "mismatch": mismatch,
            "confidence": confidence,
            "top_similarity": top_scores[:, 0],
            "score_margin": score_margin,
            "same_code": same_code,
            "mean_selected_age": mean_selected_age,
            "selected_code_ids": selected_code_ids,
            "attention": attention,
        }
        return qmem_loss, diagnostics

    def sparse_code_consistency(
        self,
        current_prequant,
        selected_code_ids,
        attention,
        confidence,
        quantizer,
    ):
        """Sparse CE against the propagated previous code-ID distribution."""
        batch, channels, height, width = current_prequant.shape
        flat = current_prequant.permute(0, 2, 3, 1).contiguous().view(-1, channels)
        selected_flat = selected_code_ids.permute(0, 2, 3, 1).contiguous().view(-1, TOP_K)
        attention_flat = attention.permute(0, 2, 3, 1).contiguous().view(-1, TOP_K).detach()

        # Detach codebook embeddings: update current encoder/pre-quant path only.
        embedding = quantizer.embedding.detach()
        losses = []

        for start in range(0, flat.shape[0], CODE_LOGIT_CHUNK):
            end = min(start + CODE_LOGIT_CHUNK, flat.shape[0])
            z_chunk = flat[start:end]
            distances = (
                z_chunk.square().sum(dim=1, keepdim=True)
                - 2.0 * (z_chunk @ embedding.t())
                + embedding.square().sum(dim=1).unsqueeze(0)
            )
            log_probs = F.log_softmax(
                -distances / CODE_ASSIGN_TEMPERATURE, dim=1
            )
            target_ids = selected_flat[start:end]
            target_weights = attention_flat[start:end]
            selected_log_probs = torch.gather(log_probs, 1, target_ids)
            losses.append(-(target_weights * selected_log_probs).sum(dim=1))

        loss_map = torch.cat(losses).view(batch, height, width)
        confidence_detached = confidence.detach()
        return (
            confidence_detached * loss_map
        ).sum() / (confidence_detached.sum() + 1e-6)

    def forward_sequence(self, current, previous_sequence, compute_code_loss=True):
        if previous_sequence.shape[1] > self.memory_frames:
            previous_sequence = previous_sequence[:, -self.memory_frames:]

        previous_q, previous_indices = self.encode_previous_codes(previous_sequence)
        current_outputs = self.encode_current(current)
        qmem_loss, diagnostics = self.propagate_previous_codes(
            current_outputs["q_bottom_st"], previous_q, previous_indices
        )
        if compute_code_loss:
            code_loss = self.sparse_code_consistency(
                current_prequant=current_outputs["prequant_bottom"],
                selected_code_ids=diagnostics["selected_code_ids"],
                attention=diagnostics["attention"],
                confidence=diagnostics["confidence"],
                quantizer=self.base.quantizer_bottom,
            )
        else:
            code_loss = torch.zeros((), device=current.device)

        current_outputs["qmem_loss"] = qmem_loss
        current_outputs["code_loss"] = code_loss
        current_outputs["diagnostics"] = diagnostics
        return current_outputs


# Shape and gradient sanity check
seed_everything(BASE_SEED)
_probe = StrictBottomCodeCAMPA(
    make_base_model(CHRISTIAN_CKPT), memory_frames=2, confidence_mode="margin"
).eval()
with torch.no_grad():
    _current = torch.zeros(1, 3, MODEL_SIZE, MODEL_SIZE, device=DEVICE)
    _previous = torch.zeros(1, 2, 3, MODEL_SIZE, MODEL_SIZE, device=DEVICE)
    _out = _probe.forward_sequence(_current, _previous)
print("Sanity reconstruction:", tuple(_out["reconstruction"].shape))
print("Sanity mismatch:", tuple(_out["diagnostics"]["mismatch"].shape))
print("Sanity selected IDs:", tuple(_out["diagnostics"]["selected_code_ids"].shape))
print("Sanity qmem/code losses:", float(_out["qmem_loss"]), float(_out["code_loss"]))
del _probe, _current, _previous, _out
gc.collect()
torch.cuda.empty_cache()


Sanity reconstruction: (1, 3, 256, 256)
Sanity mismatch: (1, 64, 64)
Sanity selected IDs: (1, 2, 64, 64)
Sanity qmem/code losses: 1.6601262586846133e-06 1.9169464111328125


In [7]:
# ============================================================
# 6) Christian objective plus v2 losses
# ============================================================


def current_crops(images_512):
    return torch.cat([get_corner_crops(image, CROP_SIZE) for image in images_512], dim=0)


def compute_objective(model, previous_512, current_512, run_config):
    current_full = F.interpolate(
        current_512, size=(MODEL_SIZE, MODEL_SIZE), mode="bilinear", align_corners=False
    )

    use_qmem = run_config["kind"] == "qmem"
    if use_qmem:
        batch, frames, channels, height, width = previous_512.shape
        previous_flat = previous_512.view(batch * frames, channels, height, width)
        previous_flat = F.interpolate(
            previous_flat, size=(MODEL_SIZE, MODEL_SIZE), mode="bilinear", align_corners=False
        )
        previous_full = previous_flat.view(
            batch, frames, channels, MODEL_SIZE, MODEL_SIZE
        )

        outputs = model.forward_sequence(
            current_full,
            previous_full,
            compute_code_loss=float(run_config["lambda_code"]) > 0.0,
        )
        recon = outputs["reconstruction"]
        q_loss_full = outputs["vq_loss"]
        qmem_loss = outputs["qmem_loss"]
        code_loss = outputs["code_loss"]
        diagnostics = outputs["diagnostics"]
        base = model.base
        perplexity_top = outputs["perplexity_top"]
        perplexity_bottom = outputs["perplexity_bottom"]
    else:
        recon, q_loss_full = model(current_full)
        zero = torch.zeros((), device=current_full.device)
        qmem_loss = zero
        code_loss = zero
        base = model
        perplexity_top = zero
        perplexity_bottom = zero
        latent_h = MODEL_SIZE // 4
        blank = torch.zeros(current_full.shape[0], latent_h, latent_h, device=current_full.device)
        diagnostics = {
            "mismatch": blank,
            "confidence": blank,
            "top_similarity": blank,
            "score_margin": blank,
            "same_code": blank,
            "mean_selected_age": blank,
        }

    crops = current_crops(current_512).to(current_full.device)
    crop_recon, q_loss_crops = base(crops)

    loss_rec = F.mse_loss(recon, current_full)
    loss_crops = F.mse_loss(crop_recon, crops)
    recon_up = F.interpolate(
        recon, size=(IMG_SIZE_TRAIN, IMG_SIZE_TRAIN), mode="bilinear", align_corners=False
    )
    crop_reassembled = reassemble_crops_to_full_fixed(crop_recon, IMG_SIZE_TRAIN)
    loss_cons = F.mse_loss(crop_reassembled, recon_up)
    loss_vq = q_loss_full + q_loss_crops

    loss_doao = (
        LAMBDA_REC * loss_rec
        + LAMBDA_CROPS * loss_crops
        + LAMBDA_CONS * loss_cons
        + LAMBDA_VQ * loss_vq
    )

    legal_pixel_error = torch.mean(torch.abs(current_full - recon), dim=1)
    clean_excess = F.relu(legal_pixel_error - CLEAN_TAIL_THRESHOLD)
    clean_tail_loss = clean_excess.mean()
    clean_tail_fraction = (
        legal_pixel_error > CLEAN_TAIL_THRESHOLD
    ).float().mean()

    weighted_qmem = float(run_config["lambda_qmem"]) * qmem_loss
    weighted_code = float(run_config["lambda_code"]) * code_loss
    weighted_clean = float(run_config["lambda_clean"]) * clean_tail_loss
    loss_total = loss_doao + weighted_qmem + weighted_code + weighted_clean

    return {
        "loss_total": loss_total,
        "loss_doao": loss_doao,
        "loss_rec": loss_rec,
        "loss_crops": loss_crops,
        "loss_cons": loss_cons,
        "loss_vq": loss_vq,
        "loss_qmem": qmem_loss,
        "loss_code": code_loss,
        "loss_clean_tail": clean_tail_loss,
        "weighted_qmem": weighted_qmem,
        "weighted_code": weighted_code,
        "weighted_clean_tail": weighted_clean,
        "clean_tail_fraction": clean_tail_fraction,
        "mean_mismatch": diagnostics["mismatch"].mean(),
        "mean_confidence": diagnostics["confidence"].mean(),
        "mean_top_similarity": diagnostics["top_similarity"].mean(),
        "mean_score_margin": diagnostics["score_margin"].mean(),
        "same_code_rate": diagnostics["same_code"].mean(),
        "mean_selected_age": diagnostics["mean_selected_age"].mean(),
        "perplexity_top": perplexity_top,
        "perplexity_bottom": perplexity_bottom,
    }


def accumulate(sums, metrics, batch_size):
    for key, value in metrics.items():
        if torch.is_tensor(value):
            value = float(value.detach().cpu())
        sums[key] = sums.get(key, 0.0) + float(value) * batch_size
    sums["n"] = sums.get("n", 0) + int(batch_size)


def finalize(sums):
    n = max(int(sums.get("n", 0)), 1)
    return {key: value / n for key, value in sums.items() if key != "n"}


def run_epoch(model, loader, run_config, optimizer=None):
    training = optimizer is not None
    model.train(training)
    sums = {}
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for previous_512, current_512, _, _, _ in tqdm(
            loader,
            leave=False,
            desc=(f"train {run_config['name']}" if training else f"validation {run_config['name']}"),
        ):
            previous_512 = previous_512.to(DEVICE, non_blocking=True)
            current_512 = current_512.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            metrics = compute_objective(model, previous_512, current_512, run_config)
            if training:
                metrics["loss_total"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            accumulate(sums, metrics, current_512.shape[0])

    return finalize(sums)


In [8]:
# ============================================================
# 7) Independent training runs and checkpoint selection
# ============================================================


def save_checkpoint(path, model, run_config, metadata):
    state = model.base.state_dict() if run_config["kind"] == "qmem" else model.state_dict()
    torch.save(
        {
            "kind": run_config["kind"],
            "model_state_dict": state,
            "run_config": run_config,
            "metadata": metadata,
        },
        path,
    )


def load_checkpoint(path):
    try:
        return torch.load(str(path), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=DEVICE)


def make_model_for_config(run_config):
    base = make_base_model(CHRISTIAN_CKPT)
    if run_config["kind"] == "qmem":
        return StrictBottomCodeCAMPA(
            base,
            memory_frames=run_config["memory_frames"],
            confidence_mode=run_config["confidence_mode"],
        ).to(DEVICE)
    return base.to(DEVICE)


def train_one_run(run_config):
    run_name = run_config["name"]
    run_dir = CKPT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / "best.pth"
    last_path = run_dir / "last.pth"
    log_path = OUT_DIR / f"epoch_log_{run_name}.csv"

    if REUSE_EXISTING_CHECKPOINTS and best_path.exists():
        saved = load_checkpoint(best_path)
        return best_path, saved.get("metadata", {})

    seed_everything(BASE_SEED)
    sequence_t = max(1, int(run_config["memory_frames"]))
    train_loader = make_train_loader(BASE_SEED, sequence_t)
    val_loader = make_val_loader(sequence_t)
    model = make_model_for_config(run_config)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_VQVAE)

    best_value = float("inf")
    best_epoch = 0
    stale = 0
    rows = []

    for epoch in range(1, EPOCHS + 1):
        print("\n" + "=" * 100)
        print(run_name, "epoch", epoch, "/", EPOCHS)

        train_metrics = run_epoch(model, train_loader, run_config, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, run_config, optimizer=None)

        row = {
            "run": run_name,
            "epoch": epoch,
            **{key: value for key, value in run_config.items() if key != "name"},
        }
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        rows.append(row)
        pd.DataFrame(rows).to_csv(log_path, index=False)

        print("TRAIN:", train_metrics)
        print("VALIDATION:", val_metrics)

        selection = val_metrics["loss_total"]
        metadata = {
            "run_name": run_name,
            "epoch": epoch,
            "selection_metric": "val_loss_total",
            "val_loss_total": float(val_metrics["loss_total"]),
            "val_loss_doao": float(val_metrics["loss_doao"]),
            "val_loss_qmem": float(val_metrics["loss_qmem"]),
            "val_loss_code": float(val_metrics["loss_code"]),
            "val_loss_clean_tail": float(val_metrics["loss_clean_tail"]),
            "train_loss_total": float(train_metrics["loss_total"]),
            "train_loss_doao": float(train_metrics["loss_doao"]),
            "train_loss_qmem": float(train_metrics["loss_qmem"]),
            "train_loss_code": float(train_metrics["loss_code"]),
            "train_loss_clean_tail": float(train_metrics["loss_clean_tail"]),
        }

        save_checkpoint(last_path, model, run_config, metadata)
        if selection < best_value - EARLY_STOPPING_MIN_DELTA:
            best_value = selection
            best_epoch = epoch
            stale = 0
            save_checkpoint(best_path, model, run_config, metadata)
            print("Saved new best checkpoint:", best_path)
        else:
            stale += 1

        if stale >= EARLY_STOPPING_PATIENCE:
            print("Early stopping at epoch", epoch)
            break

    saved = load_checkpoint(best_path)
    metadata = saved.get("metadata", {})
    metadata["best_epoch"] = best_epoch

    del model, optimizer, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return best_path, metadata


RUN_REGISTRY = {
    "christian_original_simple": {
        "kind": "christian_original",
        "checkpoint": str(CHRISTIAN_CKPT),
        "best_epoch": 0,
        "memory_frames": 0,
        "confidence_mode": "none",
        "lambda_qmem": np.nan,
        "lambda_code": np.nan,
        "lambda_clean": np.nan,
    }
}

if RUN_TRAINING:
    for run_config in RUN_CONFIGS:
        checkpoint_path, metadata = train_one_run(run_config)
        RUN_REGISTRY[run_config["name"]] = {
            **run_config,
            "checkpoint": str(checkpoint_path),
            "best_epoch": metadata.get("epoch", np.nan),
            "best_val_total": metadata.get("val_loss_total", np.nan),
            "best_val_doao": metadata.get("val_loss_doao", np.nan),
        }

with open(OUT_DIR / "run_registry.json", "w") as handle:
    json.dump(RUN_REGISTRY, handle, indent=2, default=str)
print(json.dumps(RUN_REGISTRY, indent=2, default=str))



continued_baseline_corrected_crop epoch 1 / 10


train continued_baseline_corrected_crop:   0%|          | 0/898 [00:00<?, ?it/s]

validation continued_baseline_corrected_crop:   0%|          | 0/213 [00:00<?, ?it/s]

TRAIN: {'loss_total': 0.0034501537839954992, 'loss_doao': 0.0034501537839954992, 'loss_rec': 0.0013500585288352233, 'loss_crops': 0.000873472467387927, 'loss_cons': 0.0007847968311373152, 'loss_vq': 0.00044182592539849033, 'loss_qmem': 0.0, 'loss_code': 0.0, 'loss_clean_tail': 0.00042868112473565424, 'weighted_qmem': 0.0, 'weighted_code': 0.0, 'weighted_clean_tail': 0.0, 'clean_tail_fraction': 0.0097026681461584, 'mean_mismatch': 0.0, 'mean_confidence': 0.0, 'mean_top_similarity': 0.0, 'mean_score_margin': 0.0, 'same_code_rate': 0.0, 'mean_selected_age': 0.0, 'perplexity_top': 0.0, 'perplexity_bottom': 0.0}
VALIDATION: {'loss_total': 0.0023957933819760842, 'loss_doao': 0.0023957933819760842, 'loss_rec': 0.000966786538175095, 'loss_crops': 0.0006553241205870937, 'loss_cons': 0.00045568597848832465, 'loss_vq': 0.00031799672446232815, 'loss_qmem': 0.0, 'loss_code': 0.0, 'loss_clean_tail': 0.0002770613766966075, 'weighted_qmem': 0.0, 'weighted_code': 0.0, 'weighted_clean_tail': 0.0, 'clean

train continued_baseline_corrected_crop:   0%|          | 0/898 [00:00<?, ?it/s]

...

validation strict_qmem_v2_T2K2_full:   0%|          | 0/205 [00:00<?, ?it/s]

TRAIN: {'loss_total': 0.003537695925537064, 'loss_doao': 0.0029956486193470438, 'loss_rec': 0.0012070463620640435, 'loss_crops': 0.0008315040160585934, 'loss_cons': 0.0005617388582861545, 'loss_vq': 0.00039535937113681835, 'loss_qmem': 0.002831529073554007, 'loss_code': 3.6834486862124525, 'loss_clean_tail': 0.0003212599093896757, 'weighted_qmem': 0.00014157645598497623, 'weighted_code': 0.0003683448597310079, 'weighted_clean_tail': 3.212599154212903e-05, 'clean_tail_fraction': 0.007883545177023459, 'mean_mismatch': 0.0031895667210801943, 'mean_confidence': 0.684574223127106, 'mean_top_similarity': 0.9955132140014092, 'mean_score_margin': 0.002481725969301494, 'same_code_rate': 0.5482084058440952, 'mean_selected_age': 1.0001602480396579, 'perplexity_top': 4.417272090911865, 'perplexity_bottom': 21.43055067748659}
VALIDATION: {'loss_total': 0.0027502169903658867, 'loss_doao': 0.002297707831999344, 'loss_rec': 0.000922157436479065, 'loss_crops': 0.0006487846723615541, 'loss_cons': 0.0004

train strict_qmem_v2_T2K2_full:   0%|          | 0/838 [00:00<?, ?it/s]

validation strict_qmem_v2_T2K2_full:   0%|          | 0/205 [00:00<?, ?it/s]

TRAIN: {'loss_total': 0.0034575992071693533, 'loss_doao': 0.002910705694509406, 'loss_rec': 0.0011806683110464759, 'loss_crops': 0.0008117381678883038, 'loss_cons': 0.0005468661878391473, 'loss_vq': 0.00037143301697688166, 'loss_qmem': 0.0026212745393682287, 'loss_code': 3.8497866721393885, 'loss_clean_tail': 0.0003085112263304347, 'weighted_qmem': 0.00013106372944514827, 'weighted_code': 0.0003849786574645432, 'weighted_clean_tail': 3.085112317466671e-05, 'clean_tail_fraction': 0.007616769238776967, 'mean_mismatch': 0.00297043500003725, 'mean_confidence': 0.6704017538282543, 'mean_top_similarity': 0.9958235313373383, 'mean_score_margin': 0.0024291500606996538, 'same_code_rate': 0.5290144621062257, 'mean_selected_age': 1.0001156910315099, 'perplexity_top': 4.417272090911865, 'perplexity_bottom': 22.744876671349978}
VALIDATION: {'loss_total': 0.002713568803856561, 'loss_doao': 0.0022500886464427807, 'loss_rec': 0.0009069809311235003, 'loss_crops': 0.0006308166803215018, 'loss_cons': 0.0

In [9]:
# ============================================================
# 8) Exact successful baseline evaluation protocol
# ============================================================

infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_INFER, IMG_SIZE_INFER)),
    transforms.ToTensor(),
])

# Intentionally retained from the successful strict evaluator.
mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_INFER, IMG_SIZE_INFER)),
    transforms.ToTensor(),
])
STRUCT = generate_binary_structure(2, 2)


def video_dirs(root):
    return sorted([path for path in Path(root).iterdir() if path.is_dir()], key=natural_key)


def load_rgb(path):
    return infer_transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)


def map_numpy(x, size=IMG_SIZE_INFER):
    if x.ndim == 3:
        x = x.unsqueeze(1)
    return F.interpolate(
        x, size=(size, size), mode="bilinear", align_corners=False
    )[0, 0].detach().cpu().numpy()


def load_run_model(info):
    kind = info["kind"]
    checkpoint = Path(info["checkpoint"])
    if kind == "christian_original":
        return make_base_model(checkpoint).eval()

    saved = load_checkpoint(checkpoint)
    base = VQVAE2().to(DEVICE)
    base.load_state_dict(saved["model_state_dict"])
    if kind == "qmem":
        return StrictBottomCodeCAMPA(
            base,
            memory_frames=int(info["memory_frames"]),
            confidence_mode=info["confidence_mode"],
        ).eval()
    return base.eval()


@torch.no_grad()
def sequence_maps(model, kind, current_x, history):
    if kind == "qmem" and len(history) > 0:
        needed = int(model.memory_frames)
        selected = list(history)[-needed:]
        while len(selected) < needed:
            selected.insert(0, selected[0])
        previous = torch.stack([tensor[0] for tensor in selected], dim=0).unsqueeze(0)
        output = model.forward_sequence(
            current_x, previous, compute_code_loss=False
        )
        recon = output["reconstruction"]
        diagnostics = output["diagnostics"]
    else:
        base = model.base if kind == "qmem" else model
        recon, _ = base(current_x)
        latent_h = current_x.shape[-2] // 4
        blank = torch.zeros(current_x.shape[0], latent_h, latent_h, device=current_x.device)
        diagnostics = {
            "mismatch": blank,
            "confidence": blank,
            "top_similarity": blank,
            "score_margin": blank,
            "same_code": blank,
            "mean_selected_age": blank,
        }

    error = torch.mean(torch.abs(current_x - recon), dim=1)
    return recon, error, diagnostics


def postprocess(score, threshold):
    # Copied from the successful strict-bottom evaluator.
    binary = score > threshold
    binary = binary_closing(binary, structure=STRUCT, iterations=2)
    binary = binary_opening(binary, structure=STRUCT, iterations=1)
    binary = binary_fill_holes(binary)
    labels, count = cc_label(binary)
    output = np.zeros_like(binary, dtype=np.uint8)
    for component in range(1, count + 1):
        region = labels == component
        if region.sum() >= MORPH_MIN_COMPONENT:
            output[region] = 1
    return binary_opening(output, structure=STRUCT, iterations=2).astype(np.uint8)


def mask_for_image(path):
    candidate = Path(str(path).replace("/images/", "/masks/"))
    if candidate.exists():
        return candidate
    stem = candidate.with_suffix("")
    for extension in [".png", ".jpg", ".jpeg", ".bmp"]:
        alternative = Path(str(stem) + extension)
        if alternative.exists():
            return alternative
    raise FileNotFoundError(path)


def load_mask(path):
    return (mask_transform(Image.open(path).convert("L")).squeeze().numpy() > 0.5).astype(np.uint8)


def iou_score(pred, gt):
    union = np.logical_or(pred, gt).sum()
    return float(np.logical_and(pred, gt).sum() / union) if union else 1.0


def dice_score(pred, gt):
    denominator = pred.sum() + gt.sum()
    return float(2 * np.logical_and(pred, gt).sum() / denominator) if denominator else 1.0


def precision_score(pred, gt):
    tp = np.logical_and(pred == 1, gt == 1).sum()
    fp = np.logical_and(pred == 1, gt == 0).sum()
    return float(tp / (tp + fp)) if tp + fp else (1.0 if gt.sum() == 0 else 0.0)


def recall_score(pred, gt):
    tp = np.logical_and(pred == 1, gt == 1).sum()
    fn = np.logical_and(pred == 0, gt == 1).sum()
    return float(tp / (tp + fn)) if tp + fn else 1.0


def safe_auroc(labels, scores):
    return float(roc_auc_score(labels, scores)) if len(np.unique(labels)) > 1 else float("nan")


@torch.no_grad()
def threshold_for_run(name, info):
    model = load_run_model(info)
    kind = info["kind"]
    values = []
    processed = 0

    for video in tqdm(video_dirs(VAL_AFTER_ROOT), desc=f"calibrate {name}"):
        history = deque(maxlen=max(int(info.get("memory_frames", 0)), 1))
        for path in list_images(video):
            current_x = load_rgb(path)
            _, error, _ = sequence_maps(model, kind, current_x, history)
            history.append(current_x.detach())
            error_np = gaussian_filter(error[0].cpu().numpy(), GAUSSIAN_SIGMA)
            values.append(error_np.reshape(-1))  # every pixel, float32
            processed += 1
            if MAX_THRESHOLD_FRAMES is not None and processed >= MAX_THRESHOLD_FRAMES:
                break
        if MAX_THRESHOLD_FRAMES is not None and processed >= MAX_THRESHOLD_FRAMES:
            break

    values = np.concatenate(values)
    q1, q3 = np.percentile(values, [25, 75])
    threshold = float(q3 + K_TUKEY * (q3 - q1))
    np.savez(
        OUT_DIR / f"threshold_{name}.npz",
        q1=q1,
        q3=q3,
        threshold=threshold,
        n_frames=processed,
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return threshold


THRESHOLDS = {}
if RUN_EVALUATION:
    for name, info in RUN_REGISTRY.items():
        THRESHOLDS[name] = threshold_for_run(name, info)
with open(OUT_DIR / "thresholds.json", "w") as handle:
    json.dump(THRESHOLDS, handle, indent=2)

christian_threshold = THRESHOLDS.get("christian_original_simple", float("nan"))
print("Christian threshold:", christian_threshold)
if (
    np.isfinite(christian_threshold)
    and abs(christian_threshold - EXPECTED_CHRISTIAN_THRESHOLD) > BASELINE_THRESHOLD_TOLERANCE
):
    raise RuntimeError(
        "Christian threshold protocol drift detected. "
        f"Expected about {EXPECTED_CHRISTIAN_THRESHOLD:.6f}, got {christian_threshold:.6f}."
    )


calibrate christian_original_simple:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate continued_baseline_corrected_crop:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate strict_qmem_v1_reference_T1K2_lambda0p05:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate strict_qmem_v2_T2K2_qmem_only_lambda0p02:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate strict_qmem_v2_T2K2_qmem_only_lambda0p05:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate strict_qmem_v2_T2K2_qmem_code:   0%|          | 0/32 [00:00<?, ?it/s]

calibrate strict_qmem_v2_T2K2_full:   0%|          | 0/32 [00:00<?, ?it/s]

Christian threshold: 0.1355599034577608


In [10]:
# ============================================================
# 9) Evaluate every method
# ============================================================


@torch.no_grad()
def evaluate_run(name, info, threshold):
    model = load_run_model(info)
    kind = info["kind"]
    rows = []
    labels_all = []
    scores_all = []
    start = time.time()

    for video in tqdm(video_dirs(BEFORE_ROOT), desc=f"evaluate {name}"):
        history = deque(maxlen=max(int(info.get("memory_frames", 0)), 1))
        frames = list_images(video)
        if MAX_EVAL_FRAMES is not None:
            frames = frames[:int(MAX_EVAL_FRAMES)]

        for path in frames:
            gt_path = mask_for_image(path)
            gt = load_mask(gt_path)
            current_x = load_rgb(path)
            recon, error, diagnostics = sequence_maps(model, kind, current_x, history)
            history.append(current_x.detach())

            error_np = gaussian_filter(error[0].cpu().numpy(), GAUSSIAN_SIGMA)
            pred = postprocess(error_np, threshold)

            mismatch_np = map_numpy(diagnostics["mismatch"])
            confidence_np = map_numpy(diagnostics["confidence"])
            top_similarity_np = map_numpy(diagnostics["top_similarity"])
            score_margin_np = map_numpy(diagnostics["score_margin"])
            same_code_np = map_numpy(diagnostics["same_code"])
            selected_age_np = map_numpy(diagnostics["mean_selected_age"])

            rows.append({
                "video": video.name,
                "frame": path.name,
                "method": name,
                "kind": kind,
                "memory_frames": info.get("memory_frames", 0),
                "confidence_mode": info.get("confidence_mode", "none"),
                "lambda_qmem": info.get("lambda_qmem", np.nan),
                "lambda_code": info.get("lambda_code", np.nan),
                "lambda_clean": info.get("lambda_clean", np.nan),
                "iou": iou_score(pred, gt),
                "dice": dice_score(pred, gt),
                "precision": precision_score(pred, gt),
                "recall": recall_score(pred, gt),
                "pred_pixels": int(pred.sum()),
                "gt_pixels": int(gt.sum()),
                "threshold": float(threshold),
                "mean_qmem_mismatch": float(mismatch_np.mean()),
                "mean_qmem_confidence": float(confidence_np.mean()),
                "mean_top_similarity": float(top_similarity_np.mean()),
                "mean_score_margin": float(score_margin_np.mean()),
                "same_code_rate": float(same_code_np.mean()),
                "mean_selected_age": float(selected_age_np.mean()),
                "img_path": str(path),
                "mask_path": str(gt_path),
            })
            labels_all.append(gt.reshape(-1))
            scores_all.append(error_np.reshape(-1))

    frame_df = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "method": name,
        "kind": kind,
        "memory_frames": info.get("memory_frames", 0),
        "confidence_mode": info.get("confidence_mode", "none"),
        "lambda_qmem": info.get("lambda_qmem", np.nan),
        "lambda_code": info.get("lambda_code", np.nan),
        "lambda_clean": info.get("lambda_clean", np.nan),
        "best_epoch": info.get("best_epoch", np.nan),
        "best_val_total": info.get("best_val_total", np.nan),
        "best_val_doao": info.get("best_val_doao", np.nan),
        "mean_iou": frame_df["iou"].mean(),
        "median_iou": frame_df["iou"].median(),
        "mean_dice": frame_df["dice"].mean(),
        "mean_precision": frame_df["precision"].mean(),
        "mean_recall": frame_df["recall"].mean(),
        "mean_pred_pixels": frame_df["pred_pixels"].mean(),
        "mean_gt_pixels": frame_df["gt_pixels"].mean(),
        "auroc_pixel": safe_auroc(np.concatenate(labels_all), np.concatenate(scores_all)),
        "mean_qmem_mismatch": frame_df["mean_qmem_mismatch"].mean(),
        "mean_qmem_confidence": frame_df["mean_qmem_confidence"].mean(),
        "mean_top_similarity": frame_df["mean_top_similarity"].mean(),
        "mean_score_margin": frame_df["mean_score_margin"].mean(),
        "same_code_rate": frame_df["same_code_rate"].mean(),
        "mean_selected_age": frame_df["mean_selected_age"].mean(),
        "threshold": float(threshold),
        "n_frames": len(frame_df),
        "runtime_minutes": (time.time() - start) / 60.0,
    }])

    frame_df.to_csv(OUT_DIR / f"per_frame_{name}.csv", index=False)
    summary.to_csv(OUT_DIR / f"summary_{name}.csv", index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return frame_df, summary


all_frames = []
all_summaries = []
if RUN_EVALUATION:
    for name, info in RUN_REGISTRY.items():
        frame_df, summary = evaluate_run(name, info, THRESHOLDS[name])
        all_frames.append(frame_df)
        all_summaries.append(summary)

comparison_df = pd.concat(all_frames, ignore_index=True)
comparison_summary = pd.concat(all_summaries, ignore_index=True).sort_values(
    "mean_iou", ascending=False
)
comparison_df.to_csv(OUT_DIR / "comparison_per_frame_results.csv", index=False)
comparison_summary.to_csv(OUT_DIR / "ranked_comparison_summary.csv", index=False)
print(comparison_summary.to_string(index=False))

christian_row = comparison_summary[
    comparison_summary["method"] == "christian_original_simple"
].iloc[0]
christian_iou = float(christian_row["mean_iou"])
if abs(christian_iou - EXPECTED_CHRISTIAN_IOU) > BASELINE_IOU_TOLERANCE:
    raise RuntimeError(
        "Christian mean-IoU protocol drift detected. "
        f"Expected about {EXPECTED_CHRISTIAN_IOU:.6f}, got {christian_iou:.6f}."
    )
print("Baseline protocol check passed:", christian_iou)


evaluate christian_original_simple:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate continued_baseline_corrected_crop:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate strict_qmem_v1_reference_T1K2_lambda0p05:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate strict_qmem_v2_T2K2_qmem_only_lambda0p02:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate strict_qmem_v2_T2K2_qmem_only_lambda0p05:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate strict_qmem_v2_T2K2_qmem_code:   0%|          | 0/14 [00:00<?, ?it/s]

evaluate strict_qmem_v2_T2K2_full:   0%|          | 0/14 [00:00<?, ?it/s]

                                  method               kind  memory_frames confidence_mode  lambda_qmem  lambda_code  lambda_clean  best_epoch  best_val_total  best_val_doao  mean_iou  median_iou  mean_dice  mean_precision  mean_recall  mean_pred_pixels  mean_gt_pixels  auroc_pixel  mean_qmem_mismatch  mean_qmem_confidence  mean_top_similarity  mean_score_margin  same_code_rate  mean_selected_age  threshold  n_frames  runtime_minutes
strict_qmem_v1_reference_T1K2_lambda0p05               qmem              1          legacy         0.05       0.0000           0.0          10        0.003840       0.003426  0.355086    0.358351   0.515828        0.402792     0.766015      17738.887097     9541.822581     0.843143            0.006083              0.000795             0.961383           0.003433        0.525749           0.971774   0.055115       496         0.940474
           strict_qmem_v2_T2K2_qmem_code               qmem              2          margin         0.05       0.0001        

In [11]:
# ============================================================
# 10) Paired frame/video analysis and bootstrap CIs
# ============================================================


def bootstrap_mean_ci(values, iterations=5000, seed=BASE_SEED + 99):
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(iterations, dtype=np.float64)
    for index in range(iterations):
        means[index] = rng.choice(values, size=len(values), replace=True).mean()
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


christian_frames = comparison_df[
    comparison_df["method"] == "christian_original_simple"
][["video", "frame", "iou"]].rename(columns={"iou": "christian_iou"})

paired_rows = []
per_video_rows = []
for method in comparison_summary["method"]:
    if method == "christian_original_simple":
        continue

    method_frames = comparison_df[
        comparison_df["method"] == method
    ][["video", "frame", "iou"]].rename(columns={"iou": "method_iou"})
    paired = christian_frames.merge(method_frames, on=["video", "frame"], how="inner")
    paired["delta_iou"] = paired["method_iou"] - paired["christian_iou"]
    ci_low, ci_high = bootstrap_mean_ci(paired["delta_iou"].values)

    paired_rows.append({
        "method": method,
        "mean_delta_iou_vs_christian": paired["delta_iou"].mean(),
        "median_delta_iou_vs_christian": paired["delta_iou"].median(),
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "win_fraction": float((paired["delta_iou"] > 0).mean()),
        "tie_fraction": float((paired["delta_iou"] == 0).mean()),
        "loss_fraction": float((paired["delta_iou"] < 0).mean()),
        "n_frames": len(paired),
    })

    video_summary = paired.groupby("video").agg(
        christian_mean_iou=("christian_iou", "mean"),
        method_mean_iou=("method_iou", "mean"),
        mean_delta_iou=("delta_iou", "mean"),
        n_frames=("frame", "count"),
    ).reset_index()
    video_summary["method"] = method
    per_video_rows.append(video_summary)

paired_summary = pd.DataFrame(paired_rows).sort_values(
    "mean_delta_iou_vs_christian", ascending=False
)
paired_summary.to_csv(OUT_DIR / "paired_delta_summary.csv", index=False)

per_video_summary = pd.concat(per_video_rows, ignore_index=True)
per_video_summary.to_csv(OUT_DIR / "per_video_comparison.csv", index=False)

print("\nPAIRED FRAME-LEVEL ANALYSIS:")
print(paired_summary.to_string(index=False))



PAIRED FRAME-LEVEL ANALYSIS:
                                  method  mean_delta_iou_vs_christian  median_delta_iou_vs_christian  bootstrap_ci_low  bootstrap_ci_high  win_fraction  tie_fraction  loss_fraction  n_frames
strict_qmem_v1_reference_T1K2_lambda0p05                     0.017263                       0.014976          0.004135           0.030250      0.550403           0.0       0.449597       496
           strict_qmem_v2_T2K2_qmem_code                     0.005760                       0.008542         -0.007459           0.018906      0.520161           0.0       0.479839       496
                strict_qmem_v2_T2K2_full                    -0.001244                      -0.000769         -0.015002           0.012569      0.497984           0.0       0.502016       496
strict_qmem_v2_T2K2_qmem_only_lambda0p05                    -0.003255                       0.000080         -0.017099           0.010668      0.500000           0.0       0.500000       496
strict_qmem_v2_

In [12]:
# ============================================================
# 11) Visualizations and training curves
# ============================================================


def pick_cases(frame_df):
    mean_iou = frame_df["iou"].mean()
    return {
        "worst": frame_df.sort_values("iou").iloc[0],
        "average": frame_df.iloc[(frame_df["iou"] - mean_iou).abs().argmin()],
        "best": frame_df.sort_values("iou").iloc[-1],
    }


@torch.no_grad()
def rebuild_maps(info, target_video, target_frame):
    model = load_run_model(info)
    kind = info["kind"]
    history = deque(maxlen=max(int(info.get("memory_frames", 0)), 1))
    result = None

    for path in list_images(BEFORE_ROOT / target_video):
        current_x = load_rgb(path)
        recon, error, diagnostics = sequence_maps(model, kind, current_x, history)
        history.append(current_x.detach())
        if path.name == target_frame:
            result = {
                "input": current_x[0].cpu().permute(1, 2, 0).numpy(),
                "recon": recon[0].cpu().permute(1, 2, 0).numpy(),
                "error": gaussian_filter(error[0].cpu().numpy(), GAUSSIAN_SIGMA),
                "mismatch": map_numpy(diagnostics["mismatch"]),
                "confidence": map_numpy(diagnostics["confidence"]),
                "similarity": map_numpy(diagnostics["top_similarity"]),
                "margin": map_numpy(diagnostics["score_margin"]),
                "selected_age": map_numpy(diagnostics["mean_selected_age"]),
            }
            break

    del model
    gc.collect()
    torch.cuda.empty_cache()
    if result is None:
        raise RuntimeError(f"Frame not found: {target_video}/{target_frame}")
    return result


if RUN_VISUALS:
    for name, info in RUN_REGISTRY.items():
        method_dir = VIS_DIR / name
        method_dir.mkdir(parents=True, exist_ok=True)
        method_df = comparison_df[comparison_df["method"] == name]

        for tag, row in pick_cases(method_df).items():
            maps = rebuild_maps(info, row["video"], row["frame"])
            gt = load_mask(row["mask_path"])
            pred = postprocess(maps["error"], row["threshold"])

            fig, axes = plt.subplots(2, 5, figsize=(18, 7))
            panels = [
                (maps["input"], "Input", None, 0, 1),
                (maps["recon"], "Recon", None, 0, 1),
                (maps["error"], "Reconstruction error", "hot", 0, max(float(row["threshold"]), 1e-6)),
                (maps["mismatch"], "Code-memory mismatch", "hot", 0, 2),
                (maps["confidence"], "Match confidence", "gray", 0, 1),
                (maps["similarity"], "Best cosine similarity", "viridis", -1, 1),
                (maps["margin"], "Top-1 / Top-2 margin", "viridis", 0, 0.25),
                (maps["selected_age"], "Selected frame age", "viridis", 0, max(int(info.get("memory_frames", 0)), 1)),
                (pred, "Pred", "gray", 0, 1),
                (gt, "GT", "gray", 0, 1),
            ]
            for axis, (array, title, cmap, vmin, vmax) in zip(axes.flat, panels):
                axis.imshow(array, cmap=cmap, vmin=vmin, vmax=vmax)
                axis.set_title(title)
                axis.axis("off")

            fig.suptitle(
                f"{name} | {tag} | {row['video']}/{row['frame']} | IoU={row['iou']:.3f}"
            )
            plt.tight_layout()
            plt.savefig(
                method_dir / f"{tag}_{row['video']}_{Path(row['frame']).stem}.png",
                dpi=150,
            )
            plt.close(fig)

    christian_df = comparison_df[
        comparison_df["method"] == "christian_original_simple"
    ]
    for tag, selected in pick_cases(christian_df).items():
        selected_rows = comparison_df[
            (comparison_df["video"] == selected["video"])
            & (comparison_df["frame"] == selected["frame"])
        ]
        fig, axes = plt.subplots(
            len(RUN_REGISTRY), 7, figsize=(23, 3.2 * len(RUN_REGISTRY))
        )
        if len(RUN_REGISTRY) == 1:
            axes = axes[None, :]

        for row_index, (name, info) in enumerate(RUN_REGISTRY.items()):
            row = selected_rows[selected_rows["method"] == name].iloc[0]
            maps = rebuild_maps(info, row["video"], row["frame"])
            gt = load_mask(row["mask_path"])
            pred = postprocess(maps["error"], row["threshold"])
            panels = [
                maps["input"], maps["recon"], maps["error"], maps["mismatch"],
                maps["confidence"], pred, gt,
            ]
            titles = [
                "Input", "Recon", "E_rec", "QMem mismatch", "Confidence",
                f"Pred IoU={row['iou']:.3f}", "GT",
            ]
            cmaps = [None, None, "hot", "hot", "gray", "gray", "gray"]
            for column, (array, title, cmap) in enumerate(zip(panels, titles, cmaps)):
                axes[row_index, column].imshow(array, cmap=cmap)
                axes[row_index, column].set_title(title if row_index == 0 else "")
                axes[row_index, column].axis("off")
            axes[row_index, 0].set_ylabel(name, fontsize=8)

        fig.suptitle(f"Shared {tag}: {selected['video']}/{selected['frame']}")
        plt.tight_layout()
        plt.savefig(
            VIS_DIR / f"shared_{tag}_{selected['video']}_{Path(selected['frame']).stem}.png",
            dpi=150,
        )
        plt.close(fig)

    for name in RUN_REGISTRY:
        log_file = OUT_DIR / f"epoch_log_{name}.csv"
        if not log_file.exists():
            continue
        log = pd.read_csv(log_file)
        fig, axes = plt.subplots(2, 1, figsize=(10, 9))

        for column in [
            "train_loss_total", "val_loss_total", "train_loss_doao", "val_loss_doao"
        ]:
            if column in log:
                axes[0].plot(log["epoch"], log[column], label=column)
        axes[0].set_title(name)
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Primary losses")
        axes[0].grid(alpha=0.3)
        axes[0].legend()

        for column in [
            "train_weighted_qmem", "val_weighted_qmem",
            "train_weighted_code", "val_weighted_code",
            "train_weighted_clean_tail", "val_weighted_clean_tail",
            "train_mean_confidence", "val_mean_confidence",
        ]:
            if column in log:
                axes[1].plot(log["epoch"], log[column], label=column)
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("New losses / confidence")
        axes[1].grid(alpha=0.3)
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(VIS_DIR / f"{name}_loss_curves.png", dpi=150)
        plt.close(fig)


In [13]:
# ============================================================
# 12) Package outputs
# ============================================================

manifest = {
    "method": "Strict bottom-code CAMPA v2",
    "training_data": "WS2 after frames only",
    "supervision": "self-supervised temporal consistency; no before labels",
    "matching_features": "bottom quantized VQ-VAE-2 codebook vectors only",
    "top_context_in_matching": False,
    "memory_frames_T": 2,
    "top_k_total_K": TOP_K,
    "search_radius": SEARCH_RADIUS,
    "temporal_decay": TEMPORAL_DECAY,
    "new_losses": {
        "qmem_cosine": "confidence-weighted current/propagated code-vector cosine distance",
        "code_identity": "sparse cross-entropy against propagated previous code IDs",
        "clean_tail": "penalty above Christian training-after legal error quantile",
    },
    "baseline_loss_weights": {
        "rec": LAMBDA_REC,
        "crops": LAMBDA_CROPS,
        "cons": LAMBDA_CONS,
        "vq": LAMBDA_VQ,
    },
    "run_configs": RUN_CONFIGS,
    "evaluation_protocol": {
        "threshold": "all validation-after frames, every pixel, Tukey k=4",
        "mask_resize": "same torchvision path as successful strict experiment",
        "morphology": "same closing/opening/fill/components/final-opening sequence",
        "score_precision": "float32, no test-cache quantization",
        "expected_christian_iou": EXPECTED_CHRISTIAN_IOU,
        "no_sam": True,
    },
    "memcam_fidelity": {
        "faithfully_adapted": [
            "memory bank", "cosine affinities", "temporal decay", "top-K",
            "temperature softmax", "weighted propagation", "synthetic consistency target",
        ],
        "not_the_original_memcam": [
            "no DeiT/MCTFormer backbone", "no class CAM or video labels",
            "no original trainable MAM block", "local rather than global matching",
            "vector/category consistency rather than CAM KL",
        ],
    },
    "christian_checkpoint": str(CHRISTIAN_CKPT),
    "christian_checkpoint_sha256": sha256_file(CHRISTIAN_CKPT),
    "clean_tail_calibration": CLEAN_TAIL_STATS,
    "methods": RUN_REGISTRY,
}

with open(OUT_DIR / "final_manifest.json", "w") as handle:
    json.dump(manifest, handle, indent=2, default=str)

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()
with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUT_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUT_DIR))
print("Output ZIP:", FINAL_ZIP)


Output ZIP: /kaggle/working/strict_bottom_code_campa_v2_outputs.zip
